# FAISS And Approximate Nearest Neighbor Indexes

Exhaustively scoring every stored vector gives exact results and is
simple, but it becomes expensive as collections grow. [FAISS](https://ai.meta.com/tools/faiss/) is a
library for efficient similarity search over dense vectors; its Python
interface takes vectors in, and returns results as NumPy arrays.
This notebook builds both an exact index and an
approximate nearest neighbor (ANN)
index so that the speed-accuracy trade-off is observable.

## Learning Objectives

By the end of this notebook, you should be able to:

- Add normalized chunk vectors to a FAISS index.
- Map returned vector positions back to source metadata.
- Configure an inverted-file (IVF) ANN index and its `nprobe` setting.
- Compare approximate results against an exact-search reference.

## Why An Index Exists

Exact search compares the question with every stored vector, so its cost
grows with the collection: one million 768-dimensional embeddings mean
about 768 million multiplications for every question. A vector index
organizes the stored vectors so that a search can skip most of them. An
**approximate nearest neighbor (ANN)** index uses this to return the top-k
much faster, at the risk of missing some of the true nearest neighbors
(the k most similar vectors).

FAISS only stores and searches vectors; it does not create their meaning,
so result quality still depends on the embedder. This
notebook builds an exact and an approximate FAISS index over the same
vectors and measures how often the approximate one agrees with the exact
one:

```mermaid
graph LR
    A["Embedded chunks"] --> B["FAISS index"]
    B --> C["Exact scan: IndexFlatIP"]
    B --> D["ANN partition: IndexIVFFlat"]
    Q["Embedded question"] --> C
    Q --> D
    C --> E["Reference top-k"]
    D --> F["Faster candidate top-k"]
    E --> G["Compare agreement"]
    F --> G
```

## Space Partitions And Approximation

**IVF** (inverted file) is an approximate index. During training, it
clusters a sample of vectors into `nlist` groups with k-means and keeps
the center of each group, its **centroid**. In the diagram below, each dot
is a centroid and each region, or **partition**, is the part of the space
closest to it. Every stored vector is filed into the **inverted list** of
its nearest centroid. The name comes from search engines, which keep a
list of matching documents for each word; IVF keeps a list of vectors for
each centroid. At query time, a small flat index called the **quantizer**
finds the `nprobe` centroids closest to the question, and only the vectors
in those lists are scored.

`nprobe` therefore controls how many lists are visited: a small value
scores fewer vectors but may miss a relevant neighbor filed in a list it
skipped; visiting every list scores every vector again, which makes the
search exact.

<div align="center">
  <img src="assets/03-voronoi-partitioning.png" alt="Voronoi space partitioning" width="400" style="max-width: 100%; height: auto;" />
  <p><em>Voronoi partitioning: each dot is a centroid, and each region holds the points closest to it. Source: <a href="https://commons.wikimedia.org/wiki/File:Voronoi_diagram.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>


In [ ]:
import hashlib
import json
import re
import time
from pathlib import Path
from typing import Any, Protocol, cast

import faiss
import numpy as np

documents = json.loads((Path("data") / "course_handbook.json").read_text())
chunks = [
    {
        "chunk_id": f"{doc['doc_id']}-whole",
        "doc_id": doc["doc_id"],
        "title": doc["title"],
        "section": doc["section"],
        "text": doc["text"],
    }
    for doc in documents
]
print(f"Ready to index {len(chunks)} chunks.")

The local embedding interface is repeated here so this notebook runs independently. Replace it with a trained embedding model in a production experiment, then reevaluate the index results.


In [ ]:
SYNONYMS = {
    "reimbursement": "refund",
    "reimburse": "refund",
    "charged": "payment",
    "money": "payment",
    "transfer": "payment",
    "bill": "invoice",
    "duplicated": "duplicate",
    "login": "password",
    "signin": "password",
    "phone": "device",
    "factor": "authentication",
    "throttle": "limit",
    "throttled": "limit",
    "slow": "timeout",
    "lagging": "timeout",
    "fails": "timeout",
    "failing": "timeout",
    "inference": "prediction",
    "additional": "extension",
    "thirty": "30",
    "mentoring": "mentor",
    "file": "document",
    "show": "reveal",
}
STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "can",
    "do",
    "for",
    "from",
    "how",
    "i",
    "in",
    "is",
    "my",
    "of",
    "should",
    "the",
    "to",
    "what",
    "when",
    "with",
}

Apply the explicit baseline vocabulary policy before index construction. In a production system, this hand-written synonym layer would usually give way to an evaluated trained embedding model.


In [ ]:
def tokenize(text: str) -> list[str]:
    # Map a few domain synonyms before vectors are indexed.
    words: list[str] = re.findall(r"[a-z0-9]+", text.lower())
    return [SYNONYMS.get(word, word) for word in words if word not in STOPWORDS]

In [ ]:
class HashEmbeddingModel:
    """Produce deterministic unit vectors for index demonstrations."""

    def __init__(self, dimensions: int = 2048) -> None:
        self.dimensions = dimensions

    def encode(self, texts: list[str]) -> np.ndarray:
        # FAISS expects one float32 vector row for each stored text.
        vectors = np.zeros((len(texts), self.dimensions), dtype="float32")
        for row, text in enumerate(texts):
            for token in tokenize(text):
                # Stable token slots keep the local baseline reproducible.
                slot = int.from_bytes(
                    hashlib.sha256(token.encode()).digest()[:4], "little"
                )
                vectors[row, slot % self.dimensions] += 1
        # With normalized rows, inner-product search measures cosine similarity.
        faiss.normalize_L2(vectors)
        return vectors


embedder = HashEmbeddingModel()
vectors = embedder.encode([chunk["text"] for chunk in chunks])

## Exact Similarity Search With `IndexFlatIP`

FAISS calls the dot product from notebook 02 the **inner product**, so
for normalized vectors it is again the cosine similarity. The index name
spells out how it works: *Flat* means the vectors are kept in one plain
list with no extra structure, and *IP* means they are scored by inner
product. `IndexFlatIP` compares the question with every stored vector, so
its results are exact. That makes it a trustworthy reference for
evaluating approximate indexes.

When it is imported, FAISS adds NumPy-friendly versions of its methods on
top of code generated from its C++ library, which editors cannot see.
The small `VectorIndex` protocol below names the operations used here, for
type hints only: _add_ a matrix and _search_ with a matrix.


In [ ]:
class VectorIndex(Protocol):
    """The NumPy-array interface installed by FAISS's Python wrapper."""

    def add(self, vectors: np.ndarray) -> None: ...
    def search(
        self, queries: np.ndarray, top_k: int
    ) -> tuple[np.ndarray, np.ndarray]: ...


class TrainableVectorIndex(VectorIndex, Protocol):
    nprobe: int
    ntotal: int

    def train(self, vectors: np.ndarray) -> None: ...


# Cast to the public array interface installed by the FAISS Python wrapper.
dimension = vectors.shape[1]
exact_index = cast(VectorIndex, faiss.IndexFlatIP(dimension))
exact_index.add(vectors)

The query helper restores the source metadata that FAISS positions alone cannot provide. This separation is what later makes citations and debugging possible.


In [ ]:
def retrieve(index: VectorIndex, question: str, top_k: int = 3) -> list[dict[str, Any]]:
    """Query a FAISS index and restore source metadata for valid IDs."""

    # FAISS returns positions; metadata remains in the parallel chunk list.
    scores, positions = index.search(embedder.encode([question]), top_k)
    return [
        {**chunks[position], "score": round(float(score), 4)}
        for score, position in zip(scores[0], positions[0])
        if position >= 0
    ]

In [ ]:
question = "The API replied with 429. How should my client retry?"
exact_results = retrieve(exact_index, question)
for rank, result in enumerate(exact_results, start=1):
    print(rank, result["doc_id"], result["score"], "-", result["title"])

The index returns numeric positions, not business metadata. Keeping the
`chunks` metadata mapping alongside the vectors is essential for citation,
filtering, debugging, and deletion workflows in a real vector database. Notice
that requesting three results can also return zero-score neighbors: `top_k`
is a ranking limit, not a relevance guarantee. The RAG pipeline therefore
adds an evidence threshold before generation. `top_k` is not a count
guarantee either: an IVF index that probes too few lists can return fewer
than `top_k` real results, and FAISS fills the missing slots with position
`-1`, which `retrieve` filters out through `position >= 0`.


## Build An IVF Approximate Index

IVF learns its centroids by running k-means on representative vectors.
Twelve policies are too few for meaningful centroid training, so this
example creates small deterministic perturbations only for training the partition.
The stored searchable vectors remain the twelve actual policy chunks. In a
real application, train on a representative sample of actual embeddings.


In [ ]:
# IVF needs enough training points to learn partitions in this tiny corpus.
random = np.random.default_rng(seed=7)
training_batches = []
for _ in range(12):
    # Slight perturbations expand training data without changing stored evidence.
    noisy = vectors + random.normal(0, 0.002, vectors.shape).astype("float32")
    faiss.normalize_L2(noisy)
    training_batches.append(noisy)
training_vectors = np.vstack(training_batches)
print("Training vectors:", training_vectors.shape)

In [ ]:
# Number of inverted lists, one per centroid.
nlist = 3
# The quantizer is a flat index over the centroids; it picks the lists to search.
quantizer = faiss.IndexFlatIP(dimension)
raw_ann_index = faiss.IndexIVFFlat(
    quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT
)
ann_index = cast(TrainableVectorIndex, raw_ann_index)
# Training runs k-means to learn the nlist centroids.
ann_index.train(training_vectors)
# Each vector is filed into the list of its nearest centroid.
ann_index.add(vectors)
# Start with one probed list to make approximate misses observable.
ann_index.nprobe = 1
print(f"ANN vectors stored = {ann_index.ntotal}, lists searched = {ann_index.nprobe}")

## Observe The Speed-Accuracy Trade-Off

On this tiny corpus, query time (latency) is too small to compare
meaningfully, so the experiment focuses on results instead. Treat exact
search as the reference and ask what share of its top-k results the
approximate index also returns. This notebook calls that share
**agreement**: 1.0 means the approximate top-k contains every exact
result. ANN benchmarks usually call the same number *recall*. It
measures faithfulness to exact search, but it says
nothing about whether those neighbors actually answer the question.

Results with a score of 0 are left out: they share no
words with the question, so which of them ranks first is only a
tie-break.


In [ ]:
benchmark_questions = [
    "I need a refund for a duplicate payment.",
    "I changed phones and cannot use my second factor.",
    "What should an API client do after status 429?",
    "Can I place personal email addresses in model prompts?",
    "How do I request a project extension?",
]


def top_k_agreement(question: str, nprobe: int, top_k: int = 2) -> float:
    ann_index.nprobe = nprobe
    # Treat exact search as the reference set for ANN agreement.
    # Zero-score results match nothing, so their order is only a tie-break.
    expected = {
        item["doc_id"]
        for item in retrieve(exact_index, question, top_k)
        if item["score"] > 0
    }
    observed = {
        item["doc_id"]
        for item in retrieve(ann_index, question, top_k)
        if item["score"] > 0
    }
    # Share of the exact results that the approximate index also returns.
    return len(expected & observed) / len(expected) if expected else 1.0

In [ ]:
for nprobe in [1, 2, 3]:
    agreements = [top_k_agreement(question, nprobe) for question in benchmark_questions]
    print(f"nprobe = {nprobe}: mean top-2 agreement = {np.mean(agreements):.2f}")

With one probed list, the approximate index returns on average 70% of the
exact results; probing two lists raises that to 80%. When `nprobe` equals
`nlist` (3 here), IVF scores every vector again and agreement reaches
100%, the same result as exact search. With millions of vectors, the
useful operating point is usually smaller and must be selected by
measuring retrieval quality and latency together.


## A Small Timing Harness

Timings on twelve records do not represent production performance, but
knowing how to measure repeated queries prepares you for a realistic index
benchmark.


In [ ]:
def elapsed_ms(index: VectorIndex, repeated_questions: list[str]) -> float:
    # Repeat identical work so small index overhead is measurable locally.
    start = time.perf_counter()
    for item in repeated_questions:
        retrieve(index, item, top_k=3)
    return (time.perf_counter() - start) * 1000


repeated = benchmark_questions * 100
ann_index.nprobe = 1
print("Exact milliseconds:", round(elapsed_ms(exact_index, repeated), 2))
print("IVF milliseconds:", round(elapsed_ms(ann_index, repeated), 2))

Do not over-interpret these numbers: at this scale, Python calls and
notebook timing noise dominate. The meaningful result here is that you now
have an evaluation harness that can be rerun on a larger corpus.


## Index Families And Their Trade-Offs

FAISS provides several index families. Managed vector databases expose
related choices behind configuration APIs.

| Index idea | Search behavior | Typical trade-off |
| --- | --- | --- |
| Flat exact search | Visits every vector | Accurate baseline, slower at scale |
| IVF | Visits selected learned partitions | Tune agreement with `nprobe` |
| HNSW (Hierarchical Navigable Small World) graph | Traverses neighbor links | High agreement, memory overhead |
| Product quantization (PQ) | Compresses vectors | Lower memory, possible accuracy loss |

HNSW links every stored vector to a few of its nearest neighbors and
answers a query by hopping from vector to vector toward the closest
ones. Unlike IVF, it needs no training step, and it is the default index
in Qdrant servers and single-node Chroma deployments. Qdrant's local
mode in notebook 04 does not demonstrate that server-side index.

Choose an index only after defining a quality target and query workload.
ANN is useful because it spends less search work, not because approximate
answers are acceptable by default.


## Index Persistence And Metadata Lifecycle

Vector positions are meaningful only while their metadata mapping remains
synchronized. FAISS can serialize an index, but an application must also
persist source IDs, versions, access rules, and the embedding-model
identity. Re-embedding documents with a different model without rebuilding
the index makes scores uninterpretable.


In [ ]:
# An index artifact is reproducible only alongside its vector/chunk contract.
vocabulary = json.dumps(
    {"synonyms": SYNONYMS, "stopwords": sorted(STOPWORDS)}, sort_keys=True
)
# Include chunk content, metadata, and order, not only stable IDs.
corpus = json.dumps(chunks, sort_keys=True)
index_manifest = {
    "index_type": "IndexIVFFlat",
    "metric": "inner_product",
    "normalized_vectors": True,
    "dimensions": int(dimension),
    "nlist": nlist,
    "vector_count": int(ann_index.ntotal),
    "embedding_baseline": "deterministic_hash_v1",
    # Vocabulary changes can invalidate stored vectors.
    "vocabulary_fingerprint": hashlib.sha256(vocabulary.encode()).hexdigest()[:12],
    "corpus_fingerprint": hashlib.sha256(corpus.encode()).hexdigest()[:12],
    "chunk_ids": [item["chunk_id"] for item in chunks],
}
print(json.dumps(index_manifest, indent=2))

The manifest is not a full vector database, but it records the contracts
needed to reproduce this index: how vectors are scored (inner product on
normalized vectors), how the index is partitioned, and which vocabulary
the embedder used. The corpus fingerprint also tracks chunk content,
metadata, and order. If either fingerprint changes, rebuild the index
and its metadata mapping.

A **vector database** (also called a vector
store), such as Qdrant or Chroma, wraps an index like the ones above with
everything an application needs around it: it stores each vector
together with its metadata, filters by that metadata, adds, updates, and
deletes records (when a policy changes, stale vectors must not remain
retrievable), and persists it all to disk. FAISS provides only the index,
which is why this notebook keeps the metadata in a separate `chunks` list.


## Search Quality Is Query-Dependent

A good mean agreement can hide the questions where ANN fails. With
`nprobe = 1`, IVF scores only the vectors in the list whose centroid is
closest to the question. The question below matches `api-timeout`
through the synonym map (`fails` becomes `timeout`), but it also shares
the word `request` with `password-reset`, whose list has the closer
centroid. With one probed list, the relevant policy is never scored;
probing a second list brings it back. Check hard cases like this before
finalizing a setting, and report them separately rather than hiding them
inside a mean score.


In [ ]:
difficult_question = "My prediction request takes forever and then fails."
exact = [
    (item["doc_id"], item["score"])
    for item in retrieve(exact_index, difficult_question, top_k=2)
]
print(difficult_question, "\n exact:", exact)
for nprobe in [1, 2]:
    ann_index.nprobe = nprobe
    approximate = [
        (item["doc_id"], item["score"])
        for item in retrieve(ann_index, difficult_question, top_k=2)
    ]
    print(f" approximate (nprobe = {nprobe}):", approximate)

## Exercise: Select An ANN Setting

Find the cheapest IVF setting that is as accurate as possible:

1. Use the benchmark questions and `top_k_agreement` to measure the mean top-2 agreement for each `nprobe` from 1 to `nlist`, then find the smallest value that reaches the best agreement.
2. Print both the chosen setting and its agreement score.
3. Set the index to the chosen `nprobe` and retrieve the top 2 results for the question `What follows an API rate limit response?`.


In [ ]:
# @TODO: Find the smallest nprobe that reaches the best mean top-2 agreement.


### Reference Solution


In [ ]:
# Measure the mean agreement for every nprobe setting.
scored_settings = []
for nprobe in range(1, nlist + 1):
    score = np.mean(
        [top_k_agreement(question, nprobe) for question in benchmark_questions]
    )
    scored_settings.append((nprobe, score))

# Select the least search effort that achieves the measured best agreement.
best_agreement = max(score for _, score in scored_settings)
selected_nprobe = min(
    nprobe for nprobe, score in scored_settings if score == best_agreement
)
ann_index.nprobe = selected_nprobe
assert 1 <= selected_nprobe <= nlist and 0 <= best_agreement <= 1
print(f"Selected nprobe = {selected_nprobe}, agreement = {best_agreement:.2f}")

# Use the chosen setting on a rate-limit question.
rate_limit_question = "What follows an API rate limit response?"
for result in retrieve(ann_index, rate_limit_question, top_k=2):
    print(result["doc_id"], result["score"], "-", result["title"])

Select the smallest setting with the best observed
agreement and record any remaining gap rather than pretending ANN is
lossless. Here only `nprobe = 3` reaches full agreement, and it probes
every list, so on twelve policies the approximate index saves no work. If
no approximate setting meets the required quality threshold,
keep exact search for this workload or evaluate another index design. In
production, add latency, memory, and a larger labeled query set.

In notebook 04, replace the hash baseline with trained embeddings and
use LangChain with Qdrant local mode. The integration handles vector
storage and metadata mapping; the evidence and citation checks remain ours.


## Further Reading

- FAISS: [Docs](https://faiss.ai/) / [GitHub](https://github.com/facebookresearch/faiss)
- Qdrant: [Docs](https://qdrant.tech/documentation/) / [Indexing](https://qdrant.tech/documentation/concepts/indexing/)
- Chroma: [Docs](https://docs.trychroma.com/)
- [Billion-scale similarity search with GPUs](https://arxiv.org/abs/1702.08734) - Johnson et al., 2017
- [Efficient and Robust Approximate Nearest Neighbor Search Using HNSW Graphs](https://arxiv.org/abs/1603.09320) - Malkov and Yashunin, 2016
